In [1]:
import json
import numpy as np
import pandas as pd
from scipy import stats

def analyze_llm_judge_json(json_data):
    if isinstance(json_data, str):
        data = json.loads(json_data)
    else:
        data = json_data

    dimensions = [
        "mistake_identification",
        "mistake_location",
        "providing_guidance",
        "actionability",
        "overall_score"
    ]

    # Store paired differences per dimension
    scores_dict = {dim: {"original": [], "refined": []} for dim in dimensions}

    for item in data:
        judge_scores = item["llm_judge_scores"]
        orig = judge_scores["original_scores"]
        ref = judge_scores["refined_scores"]

        for dim in dimensions[:-1]:
            scores_dict[dim]["original"].append(orig[dim]["score"])
            scores_dict[dim]["refined"].append(ref[dim]["score"])

        scores_dict["overall_score"]["original"].append(orig["overall_score"])
        scores_dict["overall_score"]["refined"].append(ref["overall_score"])

    results = []

    for dim, values in scores_dict.items():
        orig_arr = np.array(values["original"])
        ref_arr = np.array(values["refined"])
        diff = ref_arr - orig_arr
        n = len(diff)

        if n == 0:
            continue

        mean_orig = np.mean(orig_arr)
        mean_ref = np.mean(ref_arr)
        mean_diff = np.mean(diff)
        sd_diff = np.std(diff, ddof=1) if n > 1 else 0.0

        # Paired t-test
        if n > 1 and sd_diff > 0:
            t_stat, p_val_t = stats.ttest_rel(ref_arr, orig_arr)
            # Wilcoxon Signed-Rank Test (Ideal for discrete 1-3 scores)
            try:
                wilc_stat, p_val_wilc = stats.wilcoxon(ref_arr, orig_arr)
            except ValueError:
                # Handles edge case where all differences are zero
                p_val_wilc = 1.0

            # 95% Parametric Confidence Interval for Mean Difference
            sem = stats.sem(diff)
            ci_low, ci_high = stats.t.interval(0.95, df=n-1, loc=mean_diff, scale=sem)
        else:
            p_val_t, p_val_wilc = np.nan, np.nan
            ci_low, ci_high = mean_diff, mean_diff

        results.append({
            "Dimension": dim,
            "N": n,
            "Mean Orig": round(mean_orig, 2),
            "Mean Ref": round(mean_ref, 2),
            "Mean Gain": round(mean_diff, 2),
            "SD Gain": round(sd_diff, 2),
            "95% CI (Gain)": f"[{ci_low:.3f}, {ci_high:.3f}]",
            "p-value (Wilcoxon)": f"{p_val_wilc:.4e}" if not np.isnan(p_val_wilc) else "N/A",
            "p-value (Paired t)": f"{p_val_t:.4e}" if not np.isnan(p_val_t) else "N/A"
        })

    return pd.DataFrame(results)

In [2]:
# Execute on your JSON object list:
with open("outputs/llm_judge_scores.json", "r") as f:
    your_json_data = json.load(f)

df_results = analyze_llm_judge_json(your_json_data)
print(df_results.to_string(index=False))

             Dimension    N  Mean Orig  Mean Ref  Mean Gain  SD Gain  95% CI (Gain) p-value (Wilcoxon) p-value (Paired t)
mistake_identification 1688       2.08      2.82       0.74     0.94 [0.692, 0.782]        2.3261e-133        5.8036e-178
      mistake_location 1688       2.20      2.89       0.69     0.82 [0.648, 0.726]        2.9802e-145        6.4790e-198
    providing_guidance 1688       1.98      2.89       0.92     0.85 [0.877, 0.957]        4.3139e-184        3.0272e-287
         actionability 1688       2.10      2.98       0.88     0.72 [0.842, 0.910]        1.6256e-207         0.0000e+00
         overall_score 1688       2.09      2.89       0.80     0.68 [0.771, 0.836]        2.6084e-218         0.0000e+00


In [3]:
# Execute on your JSON object list:
with open("outputs/llm_judge_scores_testset.json", "r") as f:
    your_json_data = json.load(f)

df_results = analyze_llm_judge_json(your_json_data)
print(df_results.to_string(index=False))

             Dimension    N  Mean Orig  Mean Ref  Mean Gain  SD Gain  95% CI (Gain) p-value (Wilcoxon) p-value (Paired t)
mistake_identification 1010       2.07      2.83       0.76     0.93 [0.706, 0.821]         1.4837e-84        1.0881e-114
      mistake_location 1010       2.20      2.90       0.70     0.84 [0.651, 0.755]         9.3922e-87        6.3946e-119
    providing_guidance 1010       1.90      2.86       0.95     0.84 [0.901, 1.004]        4.5551e-116        6.8526e-185
         actionability 1010       2.10      2.98       0.88     0.73 [0.833, 0.924]        2.1566e-122        6.3927e-197
         overall_score 1010       2.07      2.89       0.82     0.68 [0.783, 0.866]        4.3540e-136        4.6835e-202
